> ⚠️ **Before you start:** This is a read-only course copy.
> Go to **File → Save a copy in Drive** right now, then continue working in *your* copy.
> Changes made here will not be saved.

In [ ]:
# Setup: install the course package (run this once per Colab session).
#
# Why the uninstall: the package version number never changes, so on a runtime
# that already has a copy, a plain install reports "already satisfied", skips the
# download, and leaves you running whatever code you installed last time.
#
# If you already ran an import cell before this one, use Runtime > Restart session
# afterwards. Python keeps the old module in memory even once the files are new.
!pip uninstall -q -y churn-pipeline
!pip install -q git+https://github.com/marceloacosta/churn-prediction-pipeline.git@main

# Chapter 7: LLM Integration

## Using Claude to Do the Boring Parts of ML Ops

Our pipeline has two places where a task is fundamentally about *understanding language* — something humans do effortlessly but rule-based code struggles with:

1. **Auto-Mapping (onboarding):** A new client uploads a CSV. Someone has to figure out what `MonthlyCharges`, `mrr`, or `amt_per_month` all mean. That's pattern recognition on natural language.

2. **Narrative Generation (output):** SHAP gives us `contract_type=month-to-month (+0.23)`. A business user needs: "This customer has no long-term commitment." That's translation from numbers to English.

Both tasks use Amazon Bedrock (Claude) via API, and neither one can crash the pipeline: if Bedrock fails, the call returns nothing and the caller carries on.

They are not equally harmless, though, and it is worth having the difference straight before you start. A missing narrative is cosmetic. The column reads `"N/A"` and the scores, risk tiers and SHAP reasons all ship as normal. A missing mapping stops that client dead: there is nothing to translate their columns with, and `load_mapping_config` raises on the file that is not there. Onboarding waits for a human.

So only one of these two is really optional. The **Failure Handling** section near the end works through both, including the failure that costs the most, which is not the LLM breaking but the LLM being confidently wrong.

### Cost

| Task | Typical Cost | When It Runs |
|------|-------------|-------------|
| Auto-Mapping | ~$0.01 per new client | Once, during onboarding |
| Narratives (50 customers) | ~$0.01-0.02 per batch | Each scoring run |

### What this chapter uses

Two modules do the work, one for each part of the chapter, and both have the same
three moving pieces: build a prompt, send it, parse what comes back. Everything else
imported here is the approval machinery from Chapter 1, which is what decides whether
the pipeline is allowed to act on any of it.

In [ ]:
# --- Part 1: auto-mapping. Build a prompt from a client's columns, send it, and
# turn the reply into a draft YAML somebody can review.
from churn_pipeline.llm.auto_mapping import (
    build_mapping_prompt,     # columns + sample rows + the standard schema -> one prompt
    call_bedrock_for_mapping, # sends it; returns mappings, or None if the call failed
    write_draft_yaml,         # writes the suggestions out as mapping.draft.yaml
    is_mapping_approved,      # cheap check: is there an approved mapping.yaml here?
    ColumnMapping,            # one suggestion: source column, target field, confidence, why
    _parse_mapping_response,  # Claude's JSON -> ColumnMapping objects. Underscore because
                              # it is internal; we call it directly to parse a saved reply.
)

# --- Part 2: narratives. Turn SHAP numbers for a batch of customers into English.
from churn_pipeline.llm.narrative_generator import (
    build_narrative_prompt,         # a batch of customers + their SHAP features -> one prompt
    call_bedrock_for_narratives,    # sends it; returns {customer_id: text}, or None
    parse_narrative_response,       # splits one reply back into per-customer narratives
    generate_narratives_for_batch,  # the whole loop: batching, and what to do on failure
    NarrativeRequest,               # one customer's prediction going in
    SYSTEM_PROMPT,                  # the house style every narrative has to follow
)

# --- The approval gate, from the config format built in chapter 1.
from churn_pipeline.mapping_config import (
    load_mapping_config,      # reads an approved mapping.yaml
    MappingNotApprovedError,  # ...and what it raises instead when handed a draft
)

# --- The standard field names every client's columns get translated into. This is
# the target vocabulary we hand the LLM.
from churn_pipeline.data_contract import STANDARD_SCHEMA

## First: are we calling Bedrock for real?

This chapter can run two ways.

**With credentials**, every prompt below goes to Claude on Amazon Bedrock and you
see what actually comes back, which is the point, because what comes back is not
identical every time.

**Without credentials**, the notebook falls back to saved responses and keeps going.
That fallback is a teaching device, and it is worth being precise about how far the
analogy goes. What the notebook shares with production is the detection: both LLM
functions return `None` when the call fails, and the caller branches on it. What differs
is the branch. Here it swaps in a saved answer so the chapter still has something to
explain. Production must not, because a canned narrative is indistinguishable from a
real one by the time it reaches a client. What production does instead is the last
section of this chapter.

If you want the live version, work through the **AWS credentials** setup page first:
one Bedrock API key in Colab Secrets, about five minutes. Otherwise just run on.

In [ ]:
import os


def load_aws_secrets():
    """Copy Colab Secrets into the environment, where boto3 looks for them.

    Returns the secrets it could not read, with the reason. Missing secrets are
    not an error here; the cells below fall back to saved responses.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return  # not in Colab: use whatever credentials this machine has

    missing = []
    for name in ("AWS_BEARER_TOKEN_BEDROCK", "AWS_DEFAULT_REGION"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception as e:
            missing.append((name, type(e).__name__))
    return missing


missing = load_aws_secrets() or []

LIVE = bool(os.environ.get("AWS_BEARER_TOKEN_BEDROCK"))

if LIVE:
    print(f"Credentials found. Calling Bedrock for real in {os.environ.get('AWS_DEFAULT_REGION')}.")
elif any(reason == "NotebookAccessError" for _, reason in missing):
    # The secrets exist but this notebook is not allowed to read them. Every
    # notebook needs its own toggle, including copies you saved to Drive.
    print("Your secrets exist, but this notebook cannot read them.")
    print("Open the Secrets panel (key icon) and switch Notebook access on for:")
    for name, reason in missing:
        if reason == "NotebookAccessError":
            print(f"  - {name}")
    print("Then run this cell again. Falling back to saved responses for now.")
else:
    print("No credentials found. Running on saved responses.")
    print("(A stand-in so the chapter still runs. Production returns N/A instead;")
    print(" see Failure Handling near the end.)")

## Part 1: Auto-Mapping — LLM Reads Your Column Names

### The Problem

Every company calls their data something different:

| Company A | Company B | Company C | They All Mean... |
|-----------|-----------|-----------|------------------|
| MonthlyCharges | mrr | monthly_fee | monthly_charges |
| customerID | CustID | user_id | customer_id |
| Churn | left_service | is_churned | churn_label |

A rule-based approach would need an infinite dictionary of synonyms. An LLM handles this naturally because it *understands language*.

In [ ]:
# Simulate what a new client's CSV looks like
client_columns = ["CustID", "months_active", "MonthlyFee", "TotalSpend",
                   "left_service", "plan_type", "how_they_pay", "complaints"]

sample_rows = [
    {"CustID": "USR-7590", "months_active": 12, "MonthlyFee": 59.99,
     "TotalSpend": 719.88, "left_service": "no", "plan_type": "annual",
     "how_they_pay": "credit card", "complaints": 0},
    {"CustID": "USR-3344", "months_active": 2, "MonthlyFee": 99.99,
     "TotalSpend": 199.98, "left_service": "yes", "plan_type": "monthly",
     "how_they_pay": "bank transfer", "complaints": 4},
]

print("Client's raw columns:", client_columns)
print(f"\nSample row: {sample_rows[0]}")

In [ ]:
# Build the prompt we'd send to Claude
prompt = build_mapping_prompt(client_columns, sample_rows)

print("Prompt sent to Claude (first 800 chars):")
print("=" * 60)
print(prompt[:800])
print("...")
print(f"\n(Total prompt length: {len(prompt)} characters)")

In [ ]:
# The real call. `call_bedrock_for_mapping` sends the prompt, parses the JSON that
# comes back, and returns None if anything at all went wrong: no credentials, model
# not enabled, network, a reply that isn't valid JSON. It never raises.
SAVED_MAPPING_RESPONSE = """[
    {"source_column": "CustID", "target_field": "customer_id", "confidence": "high", "reasoning": "Contains 'ID' and values look like unique identifiers"},
    {"source_column": "months_active", "target_field": "tenure_months", "confidence": "high", "reasoning": "Directly describes duration in months"},
    {"source_column": "MonthlyFee", "target_field": "monthly_charges", "confidence": "high", "reasoning": "Monthly + Fee = monthly billing amount"},
    {"source_column": "TotalSpend", "target_field": "total_charges", "confidence": "high", "reasoning": "Cumulative spending"},
    {"source_column": "left_service", "target_field": "churn_label", "confidence": "medium", "reasoning": "Binary indicator of leaving, needs value mapping yes/no to 1/0"},
    {"source_column": "plan_type", "target_field": "contract_type", "confidence": "medium", "reasoning": "Describes contract duration category"},
    {"source_column": "how_they_pay", "target_field": "payment_method", "confidence": "high", "reasoning": "Payment method description"},
    {"source_column": "complaints", "target_field": "support_tickets", "confidence": "medium", "reasoning": "Complaint count likely correlates with support interactions"}
]"""

mappings = call_bedrock_for_mapping(prompt) if LIVE else None

if mappings is None:
    if LIVE:
        print("Bedrock did not answer. Falling back to the saved response.")
        print("The warning above says what Bedrock returned. To work out what to do")
        print("about it, run the verify cell on the AWS credentials page.\n")
    mappings = _parse_mapping_response(SAVED_MAPPING_RESPONSE)
else:
    print(f"Live from Bedrock: {len(mappings)} columns mapped.\n")

print("Claude's mapping suggestions:")
print("=" * 70)
print(f"{'Client Column':<18} {'\u2192 Standard Field':<20} {'Confidence':<12} Reasoning")
print("-" * 70)
for m in mappings:
    print(f"{m.source_column:<18} \u2192 {m.target_field:<18} {m.confidence:<12} {m.reasoning[:40]}")

### The Approval Workflow

The LLM's output is a **draft**. It writes a `mapping.draft.yaml` that a human reads
before the pipeline is allowed to use it:

1. The LLM writes `mapping.draft.yaml`, with its confidence score for every column.
2. A human reads it, fixes what is wrong, and fills in what the LLM could not know.
3. The human renames it to `mapping.yaml`. The pipeline now trusts it.

**Why is a rename the approval?** Because the gate has to be something a person does on
purpose and anyone can check. No database, no approvals table, no state that can drift
out of sync with the file. You can answer "is this approved?" with `ls`.

It is enforced, not a convention. `load_mapping_config()` refuses a draft and raises
`MappingNotApprovedError`, and there is no flag to skip it. It checks two things,
because either one alone is easy to defeat by accident: the filename, and whether the
file still says `status: draft` inside. Renaming without reading does not get you past
it.

The cell below runs all three steps and leaves the file on disk so you can open it.

In [ ]:
import os

import yaml

# A real folder under the working directory, so you can go and look at these files.
# In Colab: the folder icon in the left sidebar, then client_configs/new_client.
config_dir = "client_configs/new_client"
os.makedirs(config_dir, exist_ok=True)
draft_path = os.path.join(config_dir, "mapping.draft.yaml")
approved_path = os.path.join(config_dir, "mapping.yaml")

# Step 1: the LLM writes the draft.
write_draft_yaml(mappings, "new_client", draft_path)
print(f"Step 1. The LLM wrote {draft_path}:")
print("-" * 64)
print(open(draft_path).read())

# Nothing downstream will touch it yet.
try:
    load_mapping_config(draft_path)
except MappingNotApprovedError as e:
    print(f"The pipeline refuses to load it:\n  {e}\n")

# Step 2: a human fixes what the LLM could not know. Claude saw that `left_service`
# holds "yes"/"no", but nothing in the CSV tells it the pipeline wants 1/0, so it left
# value_mappings empty. Editing that is the review. Here it is done in code; in real
# life you would open the file and type it.
config = yaml.safe_load(open(draft_path))
config["value_mappings"] = {"churn_label": {"yes": 1, "no": 0}}
config["status"] = "approved"
with open(draft_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Step 2. A human reviewed it. Two keys changed:")
print(f"        status:         'draft' -> {config['status']!r}")
print(f"        value_mappings: {{}} -> {config['value_mappings']}")

# Step 3: the rename is the gate.
print("\nStep 3. The gate, before and after the rename:")
print(f"        mapping.draft.yaml approved? {is_mapping_approved(draft_path)}   (a draft never counts)")
print(f"        mapping.yaml approved?       {is_mapping_approved(approved_path)}   (does not exist yet)")

os.rename(draft_path, approved_path)

print("        ...renamed...")
print(f"        mapping.yaml approved?       {is_mapping_approved(approved_path)}")

loaded = load_mapping_config(approved_path)
print(f"\nload_mapping_config accepts it: client_id={loaded.client_id!r}, "
      f"{len(loaded.column_mappings)} columns mapped.")

print(f"\nOpen the folder icon in the Colab sidebar and you will find one file:")
print(f"  {approved_path}")
print("There is no mapping.draft.yaml beside it, because step 3 renamed the draft")
print("rather than copying it. An approved config leaves no draft behind, which is")
print("what makes 'is there a mapping.yaml here' a complete answer. What the draft")
print("looked like is printed at the top of this cell.")

# The rename on its own is not the whole gate. Someone who renamed the file
# without reading it would leave status: draft behind, and that is still caught.
# Tried on a throwaway copy so the file above stays approved.
lazy_path = os.path.join(config_dir, "renamed_but_unread.yaml")
config["status"] = "draft"
with open(lazy_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

try:
    load_mapping_config(lazy_path)
except MappingNotApprovedError as e:
    print(f"\nRenaming without reading does not work either:\n  {e}")

os.remove(lazy_path)

### What Part 1 just did

| Cell | What happened | What came out |
|------|---------------|---------------|
| Sample CSV | A new client's column names, plus two rows of data | `client_columns`, `sample_rows` |
| Build prompt | Wrapped those in instructions and the standard schema | a prompt of about 2,000 characters |
| Call Claude | Sent it to Bedrock, or used the saved reply | 8 `ColumnMapping` objects |
| Approval | Wrote a draft, edited it, renamed it | `client_configs/new_client/mapping.yaml` |

**That last file is the actual output.** Everything the pipeline does for this client
from here on reads it to translate their column names into the standard schema. In
Chapter 1 you wrote a file like it by hand. The only thing that changed is who drafted
it, and the fact that a human still has to sign off before it counts.

## Part 2: Narrative Generation — SHAP Numbers to English

### The Problem

After scoring, we have:
```
contract_type=month-to-month (+0.23); tenure_months=2 (+0.15); support_tickets=5 (+0.18)
```

A data scientist reads that and understands. A VP of Customer Success needs:

> "This customer has no long-term commitment and has contacted support 5 times in just 2 months. Month-to-month customers with high support volume are among the most likely to leave. Consider offering a discounted annual plan."

The LLM does this translation at scale.

In [ ]:
# Build a batch of customers needing narratives
batch = [
    NarrativeRequest(
        customer_id="CUST_001",
        churn_probability=0.87,
        risk_tier="high",
        top_shap_features=[
            {"feature": "contract_type", "contribution": 0.23},
            {"feature": "support_tickets", "contribution": 0.18},
            {"feature": "tenure_months", "contribution": 0.15},
        ],
    ),
    NarrativeRequest(
        customer_id="CUST_002",
        churn_probability=0.72,
        risk_tier="high",
        top_shap_features=[
            {"feature": "monthly_charges", "contribution": 0.19},
            {"feature": "contract_type", "contribution": 0.16},
            {"feature": "tenure_months", "contribution": 0.12},
        ],
    ),
]

feature_defs = {
    "contract_type": "Whether the customer is on month-to-month, annual, or two-year plan",
    "support_tickets": "Number of times the customer contacted support",
    "tenure_months": "How long they've been a customer",
    "monthly_charges": "What they pay each month",
}

prompt = build_narrative_prompt(batch, feature_definitions=feature_defs)

print("Narrative prompt (sent to Claude):")
print("=" * 60)
print(prompt[:1200])
print("...")

In [ ]:
# Same shape as the mapping call: real when we can, saved when we can't.
SAVED_NARRATIVE_RESPONSE = """CUSTOMER_ID: CUST_001
NARRATIVE: This customer is at very high risk of leaving. They have no long-term commitment (month-to-month plan), which means there's zero friction to cancel. They've also contacted support 5 times in just 2 months — a strong signal of frustration. With only 2 months of tenure, they haven't built any loyalty yet. Consider offering a discounted annual plan to lock them in, and escalate their open support issues immediately.

CUSTOMER_ID: CUST_002
NARRATIVE: This customer is paying significantly more than average ($110/month) on a month-to-month plan. High charges without a commitment create a "why am I paying this much?" moment. They're relatively new (4 months), so they're still in the window where switching costs are low. A loyalty discount or plan review could reduce their perceived cost and extend their stay."""

narratives = call_bedrock_for_narratives(prompt) if LIVE else None

if narratives is None:
    if LIVE:
        print("Bedrock did not answer. Falling back to the saved response.\n")
    narratives = parse_narrative_response(SAVED_NARRATIVE_RESPONSE, ["CUST_001", "CUST_002"])
else:
    print(f"Live from Bedrock: {len(narratives)} narratives.\n")

print("Generated narratives:")
print("=" * 60)
for cust_id, narrative in narratives.items():
    print(f"\n{cust_id}:")
    print(f"  {narrative[:200]}..." if len(narrative) > 200 else f"  {narrative}")

## Failure Handling: Non-Blocking by Design

This is the part the saved responses above were standing in for. When a real run
loses Bedrock, nothing is substituted. The step reports that it has no answer and the
pipeline keeps going without one:

| Failure | What Happens | Client Impact |
|---------|-------------|---------------|
| Auto-mapping fails | No draft. A human writes the YAML by hand | Onboarding is blocked until they do. `load_mapping_config` raises on the missing file, so this client is not processed at all |
| Narrative generation fails | narrative_explanation = "N/A" | None. Scores, risk tiers and SHAP reasons ship as normal |

Only the second row is an enhancement. A client with no approved mapping goes nowhere,
and that is correct behaviour: the LLM took away the typing, not the requirement.

`call_bedrock_for_mapping` and `call_bedrock_for_narratives` both return `None` on any
failure rather than raising: no credentials, model not enabled, throttling, network,
a malformed reply. The caller checks for `None` and carries on.

The pipeline NEVER crashes because of an LLM failure. These are enhancements, not
dependencies.

Notice what is absent from that table: a default narrative. Degrading to `"N/A"` is
uncomfortable to look at, which is the point. A plausible sentence generated from a
template would look like the model wrote it, and nobody downstream could tell the
difference between a real explanation and filler.

### The failure this section does not cover

Everything above is about the LLM not answering. The expensive failure is the LLM
answering, fluently, and being wrong.

A mapping that sends `complaints` to `support_tickets` when the client meant something
else does not raise. It renames the column, the row counts match, validation passes, and
the model trains on a feature that means something other than its name. You find out
when the predictions are quietly bad, months later, and the mapping is the last place
anyone looks.

No `try` block catches that, which is why Part 1 ends with a human renaming a file
rather than with a confidence threshold. `confidence: high` is the model's opinion of
its own work. The rename is somebody taking responsibility for it.

The cell below forces a failure to show it. It prints a `WARNING` line first: that is
the pipeline logging what went wrong before carrying on, and seeing it is the point.
A failure that is swallowed without a trace is a failure you find out about from a
client.

In [ ]:
# Stand in for Bedrock during an outage. Handing in a client whose calls raise the
# way boto3 raises gets us a realistic warning, rather than an error about the stub
# itself.
import botocore.exceptions


class UnreachableBedrock:
    def converse(self, **kwargs):
        raise botocore.exceptions.EndpointConnectionError(
            endpoint_url="https://bedrock-runtime.us-east-1.amazonaws.com"
        )


results = generate_narratives_for_batch(
    batch,
    boto3_client=UnreachableBedrock(),
)

print("When Bedrock fails:")
print("=" * 40)
for cust_id, result in results.items():
    print(f"  {cust_id}: narrative='{result.narrative}', success={result.success}")

print("\n\u2192 Pipeline continues. Client gets SHAP reasons in top_3_reasons column.")
print("\u2192 narrative_explanation says 'N/A' instead of English paragraph.")
print("\u2192 No crash. No data loss. Just slightly less readable output.")

## The System Prompt: Controlling Claude's Output

The system prompt sets boundaries for what Claude writes. Key constraints:

- **Non-technical language** — no "SHAP values" or "feature importance"
- **Under 150 words** — concise, actionable
- **Reference specific values** — "5 support tickets" not "high support volume"
- **Plain English** — a VP should understand every word

In [ ]:
print("System prompt used for narrative generation:")
print("=" * 60)
print(SYSTEM_PROMPT)

## Key Takeaways

1. **Two LLM steps:** auto-mapping (onboarding) + narratives (output) — both are language problems
2. **Non-blocking:** if Bedrock fails, the pipeline continues without these features
3. **Human-in-the-loop:** auto-mapping produces a DRAFT that requires human approval
4. **Batch processing:** 50 customers in one prompt, not 50 API calls
5. **Cost:** pennies per run, not dollars
6. **System prompt constraints:** under 150 words, non-technical, reference specific values

Next: Chapter 8 covers the AWS architecture — how SageMaker Pipelines orchestrates all these components into a production system.

---

*Source code: `src/churn_pipeline/llm/auto_mapping.py`, `src/churn_pipeline/llm/narrative_generator.py`*  
*Tests: `tests/unit/test_auto_mapping.py`, `tests/property/test_narrative.py`, `tests/unit/test_narrative.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*